# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravindrathalari06/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-05 Section 1: build the feature vector

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Load the dataset
url = "https://raw.githubusercontent.com/Ravindrathalari06/flyrank-internship-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Features approved in ML-04
feature_fields = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier",
    "age_tier_order",
    "days_since_last_update",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "impression_tier",
    "position_tier"
]

# Separate numeric and categorical features
categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

numeric_features = [
    col for col in feature_fields
    if col not in categorical_features
]

# Preprocessing:
# - numeric missing values -> median
# - categorical missing values -> most frequent
# - categorical values -> one-hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

X = df[feature_fields]

print("Raw feature rows:", X.shape[0])
print("Raw feature columns:", X.shape[1])
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Raw feature rows: 30000
Raw feature columns: 38
Numeric features: 29
Categorical features: 9


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-05 Section 2: feature audit

feature_notes = []

categorical_set = set(categorical_features)

feature_meanings = {
    "search_volume": "Estimated search demand",
    "competition": "Competition level/value",
    "competition_level": "Categorical competition level",
    "cpc": "Cost per click",
    "content_type": "Type of content",
    "main_intent": "Primary search intent",
    "word_count": "Number of words",
    "char_count": "Number of characters",
    "impressions_90d": "Impressions over the 90-day period",
    "clicks_90d": "Clicks over the 90-day period",
    "pageviews_90d": "Pageviews over the 90-day period",
    "sessions_90d": "Sessions over the 90-day period",
    "users_90d": "Users over the 90-day period",
    "engaged_sessions_90d": "Engaged sessions over the 90-day period",
    "ai_sessions_90d": "AI-attributed sessions over the 90-day period",
    "scroll_events_90d": "Scroll events over the 90-day period",
    "days_with_impressions": "Number of days with impressions",
    "days_with_sessions": "Number of days with sessions",
    "impressions_last_30d": "Impressions in the latest 30-day period",
    "clicks_last_30d": "Clicks in the latest 30-day period",
    "sessions_last_30d": "Sessions in the latest 30-day period",
    "impressions_prev_30d": "Impressions in the previous 30-day period",
    "clicks_prev_30d": "Clicks in the previous 30-day period",
    "sessions_prev_30d": "Sessions in the previous 30-day period",
    "content_age_days": "Age of the content in days",
    "age_tier": "Categorical content-age group",
    "age_tier_order": "Ordered content-age tier",
    "days_since_last_update": "Days since the content was last updated",
    "freshness_tier": "Categorical freshness group",
    "word_count_tier": "Categorical word-count group",
    "char_count_tier": "Categorical character-count group",
    "ctr": "Click-through rate",
    "avg_position": "Average search position",
    "engagement_rate": "Engagement rate",
    "scroll_rate": "Scroll rate",
    "ai_traffic_pct": "Percentage of traffic attributed to AI",
    "impression_tier": "Categorical impression-volume group",
    "position_tier": "Categorical position group"
}

for feature in feature_fields:

    missing_count = df[feature].isna().sum()

    if feature in categorical_set:
        feature_type = "categorical"
        missing_handling = "Most-frequent imputation + one-hot encoding"
    else:
        feature_type = "numeric"
        missing_handling = "Median imputation"

    # Timing must be established before model training.
    # No explicit calendar-date columns exist in this dataset.
    available_when = "Historical value; prediction-time availability must be verified"

    feature_notes.append({
        "feature": feature,
        "meaning": feature_meanings[feature],
        "type": feature_type,
        "missing_count": int(missing_count),
        "missing_handling": missing_handling,
        "available_when": available_when
    })

feature_audit = pd.DataFrame(feature_notes)

print("Number of documented features:", len(feature_audit))
display(feature_audit)

Number of documented features: 38


,feature,meaning,type,missing_count,missing_handling,available_when
0,search_volume,Estimated search demand,numeric,2468,Median imputation,Historical value; prediction-time availability...
1,competition,Competition level/value,numeric,2468,Median imputation,Historical value; prediction-time availability...
2,competition_level,Categorical competition level,categorical,2610,Most-frequent imputation + one-hot encoding,Historical value; prediction-time availability...
3,cpc,Cost per click,numeric,2468,Median imputation,Historical value; prediction-time availability...
4,content_type,Type of content,categorical,0,Most-frequent imputation + one-hot encoding,Historical value; prediction-time availability...
5,main_intent,Primary search intent,categorical,2374,Most-frequent imputation + one-hot encoding,Historical value; prediction-time availability...
6,word_count,Number of words,numeric,7699,Median imputation,Historical value; prediction-time availability...
7,char_count,Number of characters,numeric,7699,Median imputation,Historical value; prediction-time availability...
8,impressions_90d,Impressions over the 90-day period,numeric,0,Median imputation,Historical value; prediction-time availability...
9,clicks_90d,Clicks over the 90-day period,numeric,0,Median imputation,Historical value; prediction-time availability...


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [6]:
 # ML-05 Section 3: leakage hunt

print("=== LEAKAGE HUNT ===")

# 1. Check label-derived fields
label_derived_fields = [
    "trend_direction",
    "trend_pct"
]

label_leakage = [
    field for field in label_derived_fields
    if field in feature_fields
]

print("\n1. Label-derived fields:")
print("Checked:", label_derived_fields)
print("Found in features:", label_leakage)

# 2. Check future/outcome-like feature names
future_or_outcome_keywords = [
    "trend",
    "future",
    "outcome",
    "target",
    "label"
]

future_like_features = [
    field for field in feature_fields
    if any(keyword in field.lower() for keyword in future_or_outcome_keywords)
]

print("\n2. Future/outcome-like feature names:")
print(future_like_features)

# 3. Check product/decision flags
product_decision_keywords = [
    "recommend",
    "priority",
    "decision",
    "flag",
    "action"
]

product_decision_features = [
    field for field in feature_fields
    if any(keyword in field.lower() for keyword in product_decision_keywords)
]

print("\n3. Product/decision-like feature names:")
print(product_decision_features)

# 4. Explicitly verify fields that must not be features
excluded_from_features = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used",
    "trend_direction",
    "trend_pct"
]

present_excluded_fields = [
    field for field in excluded_from_features
    if field in feature_fields
]

print("\n4. Fields checked as excluded:")
print(excluded_from_features)
print("Excluded fields found in features:", present_excluded_fields)

# 5. Overall result
if (
    not label_leakage
    and not future_like_features
    and not product_decision_features
    and not present_excluded_fields
):
    print("\nResult: No obvious name-based leakage found.")
else:
    print("\nResult: Potential leakage requires review.")

=== LEAKAGE HUNT ===

1. Label-derived fields:
Checked: ['trend_direction', 'trend_pct']
Found in features: []

2. Future/outcome-like feature names:
[]

3. Product/decision-like feature names:
[]

4. Fields checked as excluded:
['content_id', 'client_id', 'provider_used', 'model_used', 'trend_direction', 'trend_pct']
Excluded fields found in features: []

Result: No obvious name-based leakage found.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-05 Section 4: verify excluded fields

excluded_fields = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used",
    "trend_direction",
    "trend_pct"
]

print("Number of excluded fields:", len(excluded_fields))
print("\nExcluded fields and reasons:")

exclusion_reasons = {
    "content_id": "Identifier; not a meaningful predictive signal.",
    "client_id": "Client identifier; could create client-specific patterns.",
    "provider_used": "Data-generation/provider field; not a content performance signal.",
    "model_used": "Content-generation model field; not a content performance signal.",
    "trend_direction": "Observed trend outcome; using it would leak target information.",
    "trend_pct": "Trend-derived outcome measure; could leak outcome information."
}

for field in excluded_fields:
    print(f"- {field}: {exclusion_reasons[field]}")

print("\nExcluded fields present in feature set:",
      [field for field in excluded_fields if field in feature_fields])

Number of excluded fields: 6

Excluded fields and reasons:
- content_id: Identifier; not a meaningful predictive signal.
- client_id: Client identifier; could create client-specific patterns.
- provider_used: Data-generation/provider field; not a content performance signal.
- model_used: Content-generation model field; not a content performance signal.
- trend_direction: Observed trend outcome; using it would leak target information.
- trend_pct: Trend-derived outcome measure; could leak outcome information.

Excluded fields present in feature set: []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.